<a href="https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row represents one report date per pseudonymized client and content item. **Note:** The source data currently contains literal duplicate rows where all column values are identical; a `DISTINCT` or `GROUP BY` operation is required to reach a unique grain of `(report_date, client_hash_id, content_hash_id)`.

In [ ]:
import os
import duckdb
from google.colab import userdata

# Retrieve the token you saved in the Colab Secrets panel
# Ensure the toggle for "Notebook access" is turned ON in the secrets menu
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

# Initialize the DuckDB connection to the variable 'con'
con = duckdb.connect()

print("DuckDB connection 'con' successfully established and authenticated!")

DuckDB connection 'con' successfully established and authenticated!


In [ ]:
import duckdb
from google.colab import userdata

# Retrieve the token securely
hf_token = userdata.get('HF_TOKEN')

# Initialize connection
con = duckdb.connect()

# Install and load the HTTP file system extension
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Create a DuckDB secret to authenticate hf:// links
con.sql(f"CREATE SECRET hf_auth (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB is fully authenticated with Hugging Face!")

DuckDB is fully authenticated with Hugging Face!


In [ ]:
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Verified buckets: Features (gsc_clicks, gsc_impressions), Context (client_hash_id, content_hash_id, report_date)
con.sql("SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet') LIMIT 5")

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │
│    date     │         varchar         │         varchar          │      int64      │   int64    │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┤
│ 2026-06-01  │ client_3ffa76342f366962 │ content_1a6296faee432dae │               0 │          0 │
│ 2026-06-01  │ client_3ffa76342f366962 │ content_73f21e612565035a │               0 │          0 │
│ 2026-06-01  │ client_3ffa76342f366962 │ content_5a5be514ff559598 │               0 │          0 │
│ 2026-06-01  │ client_3ffa76342f366962 │ content_05b377d0c8a5cfd8 │               0 │          0 │
│ 2026-06-01  │ client_3ffa76342f366962 │ content_dc34c661d63e55a9 │               0 │          0 │
└─────────────┴─────────────────────────┴──────────────────────────┴─────────────────┴────────────┘

Features (clicks, impressions), Context (client_hash_id, content_hash_id), Excluded (url to prevent re-identification).

In [ ]:
import duckdb
from google.colab import userdata

# 1. Re-establish connection and authentication
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET IF NOT EXISTS hf_auth (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 2. Run the describe command
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Final verification: Quantifying duplicates correctly
con.sql("""
SELECT
    COUNT(*) as total_rows,
    (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'))) as unique_grain_rows,
    COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'))) as duplicate_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┬────────────────┐
│ total_rows │ unique_grain_rows │ duplicate_rows │
│   int64    │       int64       │     int64      │
├────────────┼───────────────────┼────────────────┤
│   11694072 │          11687682 │           6390 │
└────────────┴───────────────────┴────────────────┘

In [ ]:
# Investigating why (client, content, date) isn't unique.
# We will look at all columns for a known duplicate pair to find the missing grain dimension.
con.sql("""
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
WHERE client_hash_id = 'client_e00b29e582949543'
  AND content_hash_id = 'content_e3491394a9f3e2b3'
  AND report_date = '2026-06-13'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This dataset operates as an unbalanced panel where per-client history depth differs, meaning uniform historical comparisons are limited without subsetting.

In [ ]:
import duckdb
from google.colab import userdata
# Ensuring 'con' exists and is authenticated for this specific cell
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET IF NOT EXISTS hf_auth (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

con.sql("SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet') LIMIT 5")

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.